# Data pre-processing/labeling stage

Mount our data and import `pillow_heif`

In [19]:
!pip install Pillow pillow-heif

In [20]:
import os
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Do the actual preprocessing steps lowkey.

- Extract location, time_of_day, and weather
- Crop images to maintain a size of `896x1194`???, crop around the middle likely.
- Both are smallest height/width dimensions

In [21]:
def center_crop(input_path, output_path):
    img = Image.open(input_path)
    width, height = img.size

    # Smallest invidivual height/width
    target_width = 4032
    target_height = 3024

    left = (width - target_width) / 2
    top = (height - target_height) / 2
    right = (width + target_width) / 2
    bottom = (height + target_height) / 2

    cropped_img = img.crop((left, top, right, bottom))
    cropped_img.save(output_path, "JPEG")

In [22]:
import pandas as pd
import re
from PIL import Image
from pillow_heif import register_heif_opener

IMG_FOLDER = "/content/drive/My Drive/CIS_5190_group_project/Images"
OUTPUT_DIR = "/content/drive/My Drive/CIS_5190_group_project/processedImages"
IMG_LABEL_PATH = "/content/drive/My Drive/CIS_5190_group_project/img_labels.csv"

TARGET_WIDTH = 4032
TARGET_HEIGHT = 3024
img_label_df = None

# If we already have a set of existing labels, don't want to reprocess them
if os.path.exists(IMG_LABEL_PATH):
    img_label_df = pd.read_csv(IMG_LABEL_PATH)

image_names = os.listdir(IMG_FOLDER)

df_schema = {
    "file_name": [],
    "location": [],
    "time_of_day": [],
    "weather": [],
}

# Pattern to split file name into time of day, location, and weather
pattern = r"^([^_]+)_([^_]+)_([a-zA-Z]+)"
without_file_extensions = r"^([^.]+)\."
# Keep track of improperly named files that we're going to need to manually edit
wrongly_named_files = []


register_heif_opener()


# Go through each image and process it as necessary
for image_name in image_names:
    # If we've already processed the file in a previous run, skip
    if ((not (img_label_df is None)) and (image_name in img_label_df["file_name"].values)):
        continue

    input_path = os.path.join(IMG_FOLDER, image_name)
    try:
        location, tod, weather = re.match(pattern, image_name).groups()
        name = re.match(without_file_extensions, image_name).groups()[0]
    except AttributeError:
        wrongly_named_files.append(image_name)
        continue

    output_path = os.path.join(OUTPUT_DIR, name) + ".JPEG"

    img = Image.open(input_path)
    width, height = img.size
    # If our image has larger dimensions, then crop and process
    if (width >= TARGET_WIDTH) and (height >= TARGET_HEIGHT):

        df_schema["file_name"].append(image_name)
        df_schema["time_of_day"].append(tod)
        df_schema["weather"].append(weather)
        df_schema["location"].append(location)

        # Center crop our images
        center_crop(input_path, output_path)
    else:
        print(f"{image_name} not large enough")



if (not (img_label_df is None)):
    img_label_df = pd.concat((img_label_df, pd.DataFrame(df_schema)))
else:
    img_label_df = pd.DataFrame(df_schema)

img_label_df.to_csv(IMG_LABEL_PATH, index=False)
print(f"Saved to {IMG_LABEL_PATH}")

print("\nNeed to go back and edit these files:")
for img in wrongly_named_files:
    print(img)

GREGORY_SUNSET_CLOUDY_2.HEIC not large enough
GREGORY_SUNSET_CLOUDY_3.HEIC not large enough
GREGORY_SUNSET_CLOUDY_4.HEIC not large enough
GREGORY_SUNSET_CLOUDY_5.HEIC not large enough
commons_night_rain_ai.PNG not large enough
Saved to /content/drive/My Drive/CIS_5190_group_project/img_labels.csv

Need to go back and edit these files:


In [4]:
# # code to find min width/height


# import pandas as pd
# import re
# from PIL import Image
# from pillow_heif import register_heif_opener
# import numpy as np

# IMG_FOLDER = "/content/drive/My Drive/CIS_5190_group_project/Images"
# IMG_LABEL_PATH = "/content/drive/My Drive/CIS_5190_group_project/img_labels.csv"

# # Go through each image and process it as necessary
# register_heif_opener()
# image_names = os.listdir(IMG_FOLDER)

# # Something super high
# min_height = 8000000000000000000
# min_width = 80000000000000000000
# all_widths = np.zeros(shape=len(image_names))
# all_heights = np.zeros(shape=len(image_names))
# idx = 0
# for image_name in image_names:
#     fpath = os.path.join(IMG_FOLDER, image_name)
#     img = Image.open(fpath)
#     width, height = img.size

#     min_width = min(width, min_width)
#     min_height = min(height, min_height)
#     all_widths[idx] = width
#     all_heights[idx] = height
#     idx += 1



# print(f"Dimensions to look for: ({min_width}, {min_height})")

perc = 5
print(f"{perc} percentile width: {np.percentile(all_widths, 5)}")
print(f"{perc} percentile height: {np.percentile(all_heights, 5)}")

Dimensions to look for: (896, 1195)
